In [257]:
import pandas as pd
import re

In [258]:
df1 = pd.read_csv("searchdata.csv")

In [263]:
def clean_query(query):
    query = str(query).strip()  # Normalize input
    query = re.sub(r'<.*?>', '', query)   # Remove HTML tags if necessary
    query = re.sub(r'\t', ' ', query)     # Replace tabs with spaces
    query = re.sub(r'[^\w\s]', '', query) # Remove special characters
    query = re.sub(r'\s+', ' ', query)    # Replace multiple spaces with a single space
    query = query.strip()                 # Trim leading and trailing spaces

    # Define replacements using regex patterns for the first match only
    first_replacements = {
        r'(?<!\w)greatwolf lodge(?!\w)': 'great wolf lodge',
        r'(?<!\w)great wolf lodge(?!\w)': 'great wolf lodge',
        r'(?<!\w)great wolf(?!\w)': 'great wolf lodge',
        r'(?<!\w)wolf lodge(?!\w)': 'great wolf lodge',
        r'(?<!\w)oil changes(?!\w)': 'oil change',
        r'(?<!\w)cell pbone(?!\w)': 'cell phone',
        r'(?<!\w)illy bees(?!\w)': 'billy beez',
        r'(?<!\w)botox jaw(?!\w)': 'jaw botox',
        r'(?<!\w)costa rico(?!\w)': 'costa rica',
        r'(?<!\w)costa ruca(?!\w)': 'costa rica',
    }
    
    # Replace each pattern only once
    for old, new in first_replacements.items():
        if re.search(old, query):
            query = re.sub(old, new, query, count=1)  # Replace only the first match
            break  # Exit after the first match is replaced to prevent multiple replacements

    # Define replacements for all matches
    all_replacements = {
        r'\batlanata\b': 'atlanta',
        r'\bfacials\b': 'facial',
        r'\bbracelt\b': 'bracelet',
        r'\bbraclet\b': 'bracelet',
        r'\bbraclets\b': 'bracelet',
        r'\bwaxx\b': 'wax',
        r'\bmusium\b': 'museum',
        r'\bwolfe\b': 'wolf',
        r'\beax\b': 'wax',
        r'\bsalons\b': 'salon',
        r'\bpeterson\b': 'petersen',
        r'\bhotels\b': 'hotel',
        r'\bmassages\b': 'massage',
        r'\bmassagr\b': 'massage',
        r'\bcoupl\b': 'couples',
        r'\bcouple\b': 'couples',
        r'\bcoupless\b': 'couples',
        r'\bcouplesss\b': 'couples',
        r'\bmembershio\b': 'membership',
        r'\blazer\b': 'laser'
    }

    # Loop through the all replacements dictionary and perform replacements
    for old, new in all_replacements.items():
        query = re.sub(old, new, query)

    return query

In [264]:
df1['query'] = df1['query'].apply(clean_query)
df1 = df1[~df1['query'].str.isnumeric()]

texts_to_remove = ['kids', 'oil', 'family', 'chin', 'hair']
df1 = df1[~df1['query'].isin(texts_to_remove)]

df1 = df1[~df1['query'].str.contains(r'\b(sex|dildo|rose|aaa|vibrator)\b', case=False, na=False)]

df1 = df1.groupby(['query', 'division'], as_index=False).agg({'search_count': 'sum'})
result_df = df1.sort_values(['division', 'search_count'], ascending = False)

/var/folders/r0/dzfjhfp97r16q08y_8sxyg9m0000gp/T/ipykernel_25895/3211031053.py:7: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df1 = df1[~df1['query'].str.contains(r'\b(sex|dildo|rose|aaa|vibrator)\b', case=False, na=False)]


In [265]:
result_df

,query,division,search_count
47515,great wolf lodge,youngstown,185
57264,indoor waterpark,youngstown,96
72202,massage,youngstown,76
84477,oil change,youngstown,68
61413,kalahari,youngstown,62
...,...,...,...
76135,microsoft,abbotsford,5
76485,microsoft office,abbotsford,5
87826,pedicure,abbotsford,5
119502,vancouver aquarium,abbotsford,5


In [266]:
with open('/Users/zphilipp/git/suggest/data/query_by_division.txt', 'w') as file:
    for key, value in result_df.iterrows():
        if str(value['query']):
            file.write(str(value['division']) + "\t" + str(value['query']) + "\t" + str(value['search_count']) + "\n")